In [1]:
# TODO:
# -
# -
# -
# -
# -

In [2]:
# import os
# import re
# import csv
# import cv2
# import time
# import pickle
# import requests

# import numpy as np

# from datetime import date


In [3]:
import os

path_depth = "../../../"  # adjust the current working directory
if "__file__" not in globals():  # check if running in Jupyter Notebook
    # os.system("jupyter nbconvert --to script Controller.ipynb --output Controller")  # convert notebook to script
    os.system("pyuic5 -x View.ui -o View.py")  # convert UI file to Python script

In [4]:
from insightface.app import FaceAnalysis  # NOTE: this library need to import first

from View import Ui_MainWindow

from PyQt5.QtCore import *
from PyQt5.QtGui import *
from PyQt5.QtWidgets import *

import cv2
import glob
import pickle

import numpy as np

In [5]:
fa = FaceAnalysis(name="buffalo_sc", root=f"{os.getcwd()}/{path_depth}resource/utility/", providers=["CPUExecutionProvider"])
fa.prepare(ctx_id=-1, det_thresh=0.5, det_size=(640, 640))

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: c:\Users\muysengly\Desktop\repo_attendance_system_private\resource\view_controller\face_management_form/../../../resource/utility/models\buffalo_sc\det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: c:\Users\muysengly\Desktop\repo_attendance_system_private\resource\view_controller\face_management_form/../../../resource/utility/models\buffalo_sc\w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [6]:
group_paths = glob.glob(os.path.join(path_depth + "resource/database/", "*.pkl"))
group_names = [name.split("\\")[-1][:-4] for name in group_paths]
group_names

['GTR I3 ALL', 'GTR I3 Group A', 'GTR I3 Group B', 'GTR I3 Group C']

In [7]:
database = pickle.load(open(path_depth + "resource/database/" + group_names[0] + ".pkl", "rb"))
database

[]

In [8]:
class Window(Ui_MainWindow, QMainWindow):
    def __init__(self):
        super().__init__()
        self.setupUi(self)
        self.setWindowFlags(self.windowFlags() | Qt.WindowStaysOnTopHint)
        self.setWindowIcon(QIcon(f"{path_depth}resource/asset/itc_logo.png"))
        self.setWindowTitle("Face Management Form")

        self.comboBox_group.addItems(group_names)
        self.comboBox_group.setCurrentIndex(0)

        self.listView_name.setModel(QStringListModel([row[0] for row in database]))

        self.show()

In [9]:
app = QApplication([])
win = Window()

_name = ""


def on_combobox_changed():
    global group_names, database
    index = win.comboBox_group.currentIndex()
    print(f"{index} : " + win.comboBox_group.currentText())

    group_names = win.comboBox_group.currentText()
    database = pickle.load(open(path_depth + "resource/database/" + group_names + ".pkl", "rb"))

    win.listView_name.setModel(QStringListModel([row[0] for row in database]))


win.comboBox_group.currentTextChanged.connect(on_combobox_changed)


def on_button_add_clicked():

    global database

    text = win.lineEdit_name.text()
    if text:
        win.listView_name.model().insertRow(win.listView_name.model().rowCount())
        index = win.listView_name.model().index(win.listView_name.model().rowCount() - 1)
        win.listView_name.model().setData(index, text)
        print(f"Added: {text}")

        database.append([text, [None, None], [None, None]])

    print("Add button clicked")

    win.lineEdit_name.clear()
    win.lineEdit_name.setFocus()
    pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))


win.pushButton_add.clicked.connect(on_button_add_clicked)
win.lineEdit_name.returnPressed.connect(on_button_add_clicked)


def on_listview_double_clicked():
    global _name
    if win.listView_name.selectedIndexes():
        seleted = win.listView_name.selectedIndexes()[0]
        _name = seleted.data()
        print(f"{seleted.row()} : {seleted.data()}")


win.listView_name.doubleClicked.connect(on_listview_double_clicked)


def on_data_changed():
    global database
    if win.listView_name.selectedIndexes():
        selected = win.listView_name.selectedIndexes()[0]
        if selected.data() == "":
            # print("Deleted" + _name)
            win.listView_name.model().removeRow(selected.row())
            database.pop(selected.row())

        elif selected.data() != _name:
            win.listView_name.model().setData(selected, selected.data())
            # print(_name + " --> " + selected.data())

            database[selected.row()][0] = selected.data()

    pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))


win.listView_name.model().dataChanged.connect(on_data_changed)


def on_button_upload_image_1_clicked():

    if win.listView_name.selectedIndexes():
        selected = win.listView_name.selectedIndexes()[0]

        file_name, _ = QFileDialog.getOpenFileName(win, "Open Image File", "", "Images (*.jpg)")
        if file_name:

            image = np.array(cv2.imread(file_name))
            database[selected.row()][1][0] = image
            faces = fa.get(database[selected.row()][1][0])
            database[selected.row()][1][1] = faces[0].embedding

            pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))

            _image = cv2.resize(image, (win.label_image_1.width(), win.label_image_1.height()))
            q_pixmap = QPixmap.fromImage(QImage(cv2.cvtColor(_image, cv2.COLOR_BGR2RGB).data, _image.shape[1], _image.shape[0], QImage.Format.Format_RGB888))
            win.label_image_1.setPixmap(q_pixmap)


win.pushButton_upload_image_1.clicked.connect(on_button_upload_image_1_clicked)


def on_listview_single_clicked():
    if win.listView_name.selectedIndexes():
        selected = win.listView_name.selectedIndexes()[0]

        if database[selected.row()][1][0] is not None and len(database[selected.row()][1][0]) > 0:
            image = database[selected.row()][1][0]
            _image = cv2.resize(image, (win.label_image_1.width(), win.label_image_1.height()))
            q_pixmap = QPixmap.fromImage(QImage(cv2.cvtColor(_image, cv2.COLOR_BGR2RGB).data, _image.shape[1], _image.shape[0], QImage.Format.Format_RGB888))
            win.label_image_1.setPixmap(q_pixmap)

        else:
            win.label_image_1.clear()
            win.label_image_1.setText("No Image #1")


win.listView_name.clicked.connect(on_listview_single_clicked)


def on_button_take_photo_clicked():

    if win.listView_name.selectedIndexes():
        selected = win.listView_name.selectedIndexes()[0]
        win.hide()

        os.system("python " + path_depth + "resource/view_controller/take_photo_form/Controller.py")

        photo = pickle.load(open(path_depth + "resource/variable/_photo.pkl", "rb"))

        if photo is not None:

            database[selected.row()][1][0] = photo
            faces = fa.get(database[selected.row()][1][0])
            database[selected.row()][1][1] = faces[0].embedding

            pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))

            _image = cv2.resize(photo, (win.label_image_1.width(), win.label_image_1.height()))
            q_pixmap = QPixmap.fromImage(QImage(cv2.cvtColor(_image, cv2.COLOR_BGR2RGB).data, _image.shape[1], _image.shape[0], QImage.Format.Format_RGB888))
            win.label_image_1.setPixmap(q_pixmap)

        win.show()


win.pushButton_take_photo_1.clicked.connect(on_button_take_photo_clicked)


def on_button_clear_image_1_clicked():
    if win.listView_name.selectedIndexes():
        selected = win.listView_name.selectedIndexes()[0]

        database[selected.row()][1][0] = None
        database[selected.row()][1][1] = None
        win.label_image_1.clear()
        win.label_image_1.setText("No Image #1")

        pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))


win.pushButton_clear_image_1.clicked.connect(on_button_clear_image_1_clicked)


app.exec_()


database.sort(key=lambda x: x[0])
pickle.dump(database, open(path_depth + "resource/database/" + win.comboBox_group.currentText() + ".pkl", "wb"))


app = None

Added: 001
Add button clicked
Added: 002
Add button clicked
Added: 003
Add button clicked
1 : 002


In [17]:
debug = True


In [11]:
# database.sort(key=lambda x: x[0])
# pickle.dump(database, open(path_depth + "resource/database/" + group_names[0] + ".pkl", "wb"))
# [print(row[0]) for row in database]

In [12]:
# group_names[0]

In [13]:
# database = []
# pickle.dump(database, open(path_depth + "resource/database/" + group_names[0] + ".pkl", "wb"))

In [14]:
# win.listView_name.selectedIndexes()

In [15]:
# _name = ""


# def on_double_clicked():
#     global _name
#     if win.listView_group.selectedIndexes():
#         seleted = win.listView_group.selectedIndexes()[0]
#         _name = seleted.data()
#         print(f"{seleted.row()} : {seleted.data()}")


# win.listView_group.doubleClicked.connect(on_double_clicked)


# def on_data_changed():
#     global group_names
#     if win.listView_group.selectedIndexes():
#         selected = win.listView_group.selectedIndexes()[0]
#         if selected.data() == "":
#             print("Deleted" + _name)
#             win.listView_group.model().removeRow(selected.row())
#             os.remove(os.path.join(path_depth + "resource/database/", _name + ".pkl"))

#         elif selected.data() != _name:
#             win.listView_group.model().setData(selected, selected.data())
#             print(_name + " --> " + selected.data())
#             os.rename(
#                 os.path.join(path_depth + "resource/database/", _name + ".pkl"),
#                 os.path.join(path_depth + "resource/database/", selected.data() + ".pkl"),
#             )
#     group_names = win.listView_group.model().stringList()


# win.listView_group.model().dataChanged.connect(on_data_changed)


# def f_add():
#     global group_names

#     win.listView_group.clearSelection()

#     text = win.lineEdit_group_name.text()
#     if text:
#         win.listView_group.model().insertRow(win.listView_group.model().rowCount())
#         index = win.listView_group.model().index(win.listView_group.model().rowCount() - 1)
#         win.listView_group.model().setData(index, text)
#         print(f"Added: {text}")

#         pickle.dump([], open(os.path.join(path_depth + "resource/database/", text + ".pkl"), "wb"))

#     group_names = win.listView_group.model().stringList()


# win.pushButton_add.clicked.connect(f_add)